Instalaciones necesarias

In [ ]:
%pip install pettingzoo "stable-baselines3[extra]" supersuit torch matplotlib seaborn

In [ ]:
%pip install stable-baselines3[extra]

In [ ]:
%pip install supersuit

In [ ]:
%pip install numpy

Codigo para multiagente

Se desarrollo dos agentes de los cuales uno es curioso y le gusta explorar dandole recompensas por explorar y el otro es mas conversador, tiene recompensas por tener la energia arriba de 3

In [ ]:
import os
import random
import numpy as np
import gymnasium
from gymnasium import spaces

from pettingzoo import AECEnv
from pettingzoo.utils import agent_selector
from pettingzoo.utils.conversions import aec_to_parallel 

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import VecMonitor
from stable_baselines3.common.logger import configure
from stable_baselines3.common.callbacks import BaseCallback
import supersuit as ss 

# Configuración de Directorios
LOG_DIR = "logs/"
MODEL_DIR = "models/"
MODEL_NAME = "ppo_multi_agente" 
TOTAL_TIMESTEPS = 500_000  # Cantidad de saltos que realizara

os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# Definiciones del Juego 
ACCION_MOVER_N = 0
ACCION_MOVER_S = 1
ACCION_MOVER_E = 2
ACCION_MOVER_O = 3
ACCION_RECOLECTAR = 4
ACCION_COMER = 5
ACCION_DESCANSAR = 6
NUM_ACCIONES = 7

MAPA_SUELO = 0
MAPA_COMIDA = 1
MAPA_PELIGRO = 2
MAPA_DESCANSO = 3

class AgenteInterno:
    def __init__(self, id_agente, pos_x, pos_y):
        self.id_num = int(id_agente.split("_")[1]) 
        self.x = pos_x
        self.y = pos_y
        self.salud = 5
        self.hambre = 5
        self.energia = 5
        self.inventario_comida = 0
        self.esta_vivo = True
        self.celdas_visitadas = set()
        self.celdas_visitadas.add((pos_x, pos_y))

class EntornoSupervivenciaPZ(AECEnv):
    metadata = {"render_modes": ["human"], "name": "supervivencia_final", "is_parallelizable": True}

    def __init__(self, ancho=10, alto=10, max_pasos=100):
        super().__init__()
        self.ancho = ancho
        self.alto = alto
        self.max_pasos_por_episodio = max_pasos
        
        self.possible_agents = ["agente_0", "agente_1"]
        self.agents = self.possible_agents[:]
        
        self._action_space = spaces.Discrete(NUM_ACCIONES)
        
        obs_low = np.array([0, 0, 0, 0, 0, 0, 0, 0])
        obs_high = np.array([5, 5, 5, 1, self.ancho-1, self.alto-1, 3, 1])
        self._observation_space = spaces.Box(low=obs_low, high=obs_high, dtype=np.int32)
        
        self.render_mode = "human"

    def observation_space(self, agent):
        return self._observation_space
    
    def action_space(self, agent):
        return self._action_space

    def reset(self, seed=None, options=None):
        self.agents = self.possible_agents[:]
        self.rewards = {agent: 0 for agent in self.agents}
        self._cumulative_rewards = {agent: 0 for agent in self.agents}
        self.terminations = {agent: False for agent in self.agents}
        self.truncations = {agent: False for agent in self.agents}
        self.infos = {agent: {} for agent in self.agents}
        
        self.mapa_base = np.zeros((self.alto, self.ancho), dtype=int)
        self._generar_mapa()
        self.pasos_actuales = 0
        
        self.agentes_internos = {}
        for ag_id in self.agents:
            x, y = self._posicion_aleatoria_libre()
            self.agentes_internos[ag_id] = AgenteInterno(ag_id, x, y)
            
        self._agent_selector = agent_selector(self.possible_agents)
        self.agent_selection = self._agent_selector.next()

    def _generar_mapa(self):
        self.mapa_base.fill(MAPA_SUELO)
        for _ in range(int(self.ancho * self.alto * 0.15)):
            x, y = self._posicion_aleatoria_libre()
            self.mapa_base[y, x] = MAPA_COMIDA
        for _ in range(int(self.ancho * self.alto * 0.05)):
            x, y = self._posicion_aleatoria_libre()
            self.mapa_base[y, x] = MAPA_PELIGRO
        for _ in range(int(self.ancho * self.alto * 0.05)):
            x, y = self._posicion_aleatoria_libre()
            self.mapa_base[y, x] = MAPA_DESCANSO

    def _posicion_aleatoria_libre(self):
        while True:
            x = random.randint(0, self.ancho - 1)
            y = random.randint(0, self.alto - 1)
            if self.mapa_base[y, x] == MAPA_SUELO:
                return x, y

    def observe(self, agent):
        ag = self.agentes_internos[agent]
        casilla_actual = self.mapa_base[ag.y, ag.x]
        return np.array([
            ag.salud, ag.hambre, ag.energia, ag.inventario_comida,
            ag.x, ag.y, casilla_actual, ag.id_num
        ], dtype=np.int32)

    def _was_dead_step(self, action):
        agent = self.agent_selection
        if agent in self.agents:
            self.agents.remove(agent)
        
        self.terminations.pop(agent, None)
        self.truncations.pop(agent, None)
        self.rewards.pop(agent, None)
        self._cumulative_rewards.pop(agent, None)
        self.infos.pop(agent, None)
        
        self.agent_selection = self._agent_selector.next()
        while self.agent_selection not in self.agents and len(self.agents) > 0:
             self.agent_selection = self._agent_selector.next()

    def step(self, action):
        agent = self.agent_selection
        
        if self.terminations[agent] or self.truncations[agent]:
            self._was_dead_step(action)
            return

        ag = self.agentes_internos[agent]
        self.rewards[agent] = -0.01 
        
        # MOVIMIENTO Y ACCIONES 
        movimiento_realizado = False
        
        if ag.energia == 0 and action != ACCION_DESCANSAR:
            self.rewards[agent] -= 0.1
        else:
            if action != ACCION_DESCANSAR: ag.energia -= 1
            
            nuevo_x, nuevo_y = ag.x, ag.y
            if action == ACCION_MOVER_N and ag.y > 0: nuevo_y -= 1
            elif action == ACCION_MOVER_S and ag.y < self.alto - 1: nuevo_y += 1
            elif action == ACCION_MOVER_E and ag.x < self.ancho - 1: nuevo_x += 1
            elif action == ACCION_MOVER_O and ag.x > 0: nuevo_x -= 1
            
            if nuevo_x != ag.x or nuevo_y != ag.y:
                ag.x, ag.y = nuevo_x, nuevo_y
                movimiento_realizado = True

            elif action == ACCION_RECOLECTAR:
                if self.mapa_base[ag.y, ag.x] == MAPA_COMIDA:
                    if ag.inventario_comida == 0:
                        ag.inventario_comida = 1
                        self.rewards[agent] += 0.5
                        self.mapa_base[ag.y, ag.x] = MAPA_SUELO 
                    else: self.rewards[agent] -= 0.1
                else: self.rewards[agent] -= 0.05
            elif action == ACCION_COMER:
                if ag.inventario_comida == 1:
                    ag.inventario_comida = 0
                    if ag.hambre < 4:
                        self.rewards[agent] += 1.5 
                        ag.hambre = 5
                    else:
                        self.rewards[agent] -= 0.2
                        ag.hambre = 5
                else: self.rewards[agent] -= 0.1
            elif action == ACCION_DESCANSAR:
                if self.mapa_base[ag.y, ag.x] == MAPA_DESCANSO:
                    ag.energia = 5
                    self.rewards[agent] += 0.1
                else:
                    ag.energia = min(5, ag.energia + 3)

        # PERSONALIDADES (Explorador vs Conservador) ---
        if ag.id_num == 0: # Curioso
            if (ag.x, ag.y) not in ag.celdas_visitadas:
                self.rewards[agent] += 0.2
                ag.celdas_visitadas.add((ag.x, ag.y))
            elif movimiento_realizado:
                self.rewards[agent] += 0.01 

        if ag.id_num == 1: # Conservador
            if ag.energia >= 3:
                self.rewards[agent] += 0.05
            if self.mapa_base[ag.y, ag.x] == MAPA_DESCANSO:
                self.rewards[agent] += 0.1
            if movimiento_realizado:
                self.rewards[agent] -= 0.02 

        if self.pasos_actuales > 0 and self.pasos_actuales % 5 == 0:
             ag.hambre = max(0, ag.hambre - 1)
        
        if ag.hambre <= 1: self.rewards[agent] -= 0.1
        if ag.hambre == 0:
            ag.salud -= 1
            self.rewards[agent] -= 0.5
        if self.mapa_base[ag.y, ag.x] == MAPA_PELIGRO:
            ag.salud -= 1
            self.rewards[agent] -= 0.5
            
        if ag.salud <= 0:
            self.terminations[agent] = True
            self.rewards[agent] -= 1.0
            ag.esta_vivo = False
        
        self._cumulative_rewards[agent] = 0
        self.agent_selection = self._agent_selector.next()
        
        while self.agent_selection not in self.agents and len(self.agents) > 0:
             self.agent_selection = self._agent_selector.next()
        
        if len(self.agents) > 0 and self.agent_selection == self.agents[0]:
            self.pasos_actuales += 1
            
        if self.pasos_actuales >= self.max_pasos_por_episodio or len(self.agents) == 0:
            for a in self.agents:
                self.truncations[a] = True
                if self.agentes_internos[a].esta_vivo:
                    self.rewards[a] += 2.0

    def render(self):
        mapa_render = np.full((self.alto, self.ancho), " . ")
        mapa_render[self.mapa_base == MAPA_COMIDA] = " F "
        mapa_render[self.mapa_base == MAPA_PELIGRO] = " X "
        mapa_render[self.mapa_base == MAPA_DESCANSO] = " H "
        for ag_id, ag in self.agentes_internos.items():
            if ag.esta_vivo:
                idx = ag_id.split("_")[1] 
                mapa_render[ag.y, ag.x] = f" A{idx}"
            else:
                mapa_render[ag.y, ag.x] = " † "
        print("\n" + "-" * (self.ancho * 3 + 2))
        for fila in mapa_render:
            print(f"|{''.join(fila)}|")
        print("-" * (self.ancho * 3 + 2))

class MultiAgentCallback(BaseCallback):
    def __init__(self, verbose=0):
        super(MultiAgentCallback, self).__init__(verbose)

    def _on_step(self) -> bool:
        dones = self.locals['dones']
        infos = self.locals['infos']
        for i, done in enumerate(dones):
            if done:
                info = infos[i]
                if "episode" in info:
                    recompensa = info["episode"]["r"]
                    longitud = info["episode"]["l"]
                    nombre_agente = f"agente_{i}"
                    self.logger.record(f"rollout/{nombre_agente}_rew_mean", recompensa)
                    self.logger.record(f"rollout/{nombre_agente}_len_mean", longitud)
        return True

################################################################################
#                  ENTRENAMIENTO                                               #
################################################################################

env = EntornoSupervivenciaPZ(ancho=10, alto=10, max_pasos=100)
env = aec_to_parallel(env)
env = ss.black_death_v3(env)
env = ss.pettingzoo_env_to_vec_env_v1(env)
env = ss.concat_vec_envs_v1(env, num_vec_envs=1, num_cpus=1, base_class='stable_baselines3')
env = VecMonitor(env)

log_path = os.path.join(LOG_DIR, MODEL_NAME)
new_logger = configure(log_path, ["stdout", "csv", "tensorboard"])

print("Iniciando entrenamiento...")

model = PPO(
    "MlpPolicy", 
    env, 
    verbose=1, 
    learning_rate=0.0003,
    n_steps=2048,       
    batch_size=64,
    ent_coef=0.03,      
    gamma=0.995         
)
model.set_logger(new_logger)

if __name__ == "__main__":
    model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=MultiAgentCallback())
    
    print("--- Entrenamiento finalizado ---")
    model.save(os.path.join(MODEL_DIR, MODEL_NAME))
    print(f"Modelo guardado en: {MODEL_DIR}")

Logging to logs/ppo_multi_agente


c:\Users\C0okyez\AppData\Local\Programs\Python\Python313\Lib\site-packages\stable_baselines3\common\vec_env\base_vec_env.py:78: UserWarning: The `render_mode` attribute is not defined in your environment. It will be set to None.
  warnings.warn("The `render_mode` attribute is not defined in your environment. It will be set to None.")


Iniciando entrenamiento FINAL (Correcto y Exacto)...
Using cpu device
-------------------------------------
| rollout/             |            |
|    agente_0_len_mean | 31         |
|    agente_0_rew_mean | -10.26     |
|    agente_1_len_mean | 31         |
|    agente_1_rew_mean | -6.4799995 |
|    ep_len_mean       | 32.9       |
|    ep_rew_mean       | -9         |
| time/                |            |
|    fps               | 1898       |
|    iterations        | 1          |
|    time_elapsed      | 2          |
|    total_timesteps   | 4096       |
-------------------------------------
-----------------------------------------
| rollout/                |             |
|    agente_0_len_mean    | 31          |
|    agente_0_rew_mean    | -9.540001   |
|    agente_1_len_mean    | 31          |
|    agente_1_rew_mean    | -10.7300005 |
|    ep_len_mean          | 32.8        |
|    ep_rew_mean          | -8.63       |
| time/                   |             |
|    fps            